# EDA

## Prerequisite

### Import libraries

In [1]:
import duckdb
from pathlib import Path
from src.configs import DB_SAVE_PATH
from loguru import logger

import pandas as pd

### Import database

In [2]:
try:
    con = duckdb.connect(Path(DB_SAVE_PATH), read_only=True)
    logger.success("Database connected")
except duckdb.Error as err:
    logger.error(f"Unable to connect to database at {DB_SAVE_PATH}: {err}")

2026-10-22 11:17:32.034 | SUCCESS  | __main__:<module>:3 - Database connected


## Data overview

In [23]:
print("Database version:", con.execute("""
    SELECT *
    FROM version
""").fetchone()[0])

Database version: e44f186d6f06dd8452aaf54c7921ba66c961f637


### Database schema

In [31]:
db_stats = []

db_tables = con.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_type = 'BASE TABLE'
""").fetchall()

for (table,) in db_tables:
    row_count = con.execute(f"""
        SELECT COUNT(*) 
        FROM \"{table}\"
    """).fetchone()[0]

    col_count = con.execute(f"""
        SELECT COUNT(*) 
        FROM information_schema.columns 
        WHERE table_name = '{table}'
    """).fetchone()[0]

    db_stats.append(
        {"table_name": table, "row_count": row_count, "column_count": col_count}
    )

stats_df = pd.DataFrame(db_stats).sort_values(by="row_count", ascending=False).reset_index(drop=True)
display(stats_df)

,table_name,row_count,column_count
0,game_lineups,3179016,10
1,appearances,1894350,13
2,game_events,1274469,11
3,player_valuations,656301,6
4,club_games,177916,11
5,transfers,175165,10
6,games,88958,23
7,players,50149,26
8,clubs,796,17
9,national_teams,124,17


### Review of quantitative features

In [33]:
NUMERIC_TYPES = (
    r"^(U?(TINY|SMALL|BIG|HUGE)?INT(EGER)?|FLOAT|DOUBLE|REAL|DECIMAL.*|NUMERIC.*)$"
)

summaries = []
for (table, ) in db_tables:
    df = con.execute(f"""
        SELECT '{table}' AS table_name, s.*
        FROM (SUMMARIZE {table}) s
        WHERE regexp_matches(s.column_type, '{NUMERIC_TYPES}')
          AND s.column_name NOT LIKE '%!_id' ESCAPE '!'
    """).df()
    if not df.empty:
        summaries.append(df)

quantitative_features = pd.concat(summaries, ignore_index=True)
display(
    quantitative_features[
        [
            "table_name",
            "column_name",
            "column_type",
            "min",
            "max",
            "avg",
            "std",
            "q25",
            "q50",
            "q75",
            "null_percentage",
        ]
    ].sort_values(by="null_percentage").reset_index(drop=True)
)

,table_name,column_name,column_type,min,max,avg,std,q25,q50,q75,null_percentage
0,appearances,yellow_cards,INTEGER,0,2,0.14493520204819596,0.3621893845084208,0,0,0,0.00
1,appearances,red_cards,INTEGER,0,1,0.0034777100324649615,0.05886949460327004,0,0,0,0.00
2,appearances,goals,INTEGER,0,6,0.0952500857814026,0.32951475143908937,0,0,0,0.00
3,appearances,assists,INTEGER,0,6,0.07496238815424816,0.2844562854892381,0,0,0,0.00
4,appearances,minutes_played,INTEGER,0,148,68.51902235595323,30.20032758520065,46,90,90,0.00
5,clubs,squad_size,INTEGER,0,50,26.776381909547737,8.28746086095652,25,28,31,0.00
6,clubs,national_team_players,INTEGER,0,24,3.9233668341708543,4.576989944500191,0,2,6,0.00
7,clubs,foreigners_number,INTEGER,0,36,10.552763819095478,6.6873375058937565,5,10,15,0.00
8,club_games,opponent_goals,INTEGER,0,19,1.4626171901346703,1.3977599137560466,0,1,2,0.00
9,club_games,own_goals,INTEGER,0,19,1.4626171901346703,1.3977599137560475,0,1,2,0.00


#### Initial observations

##### NULL rates

- `total_market_value` in `clubs` has a 100% null rate, which means that the maintainer hasn't found any club's valuation up to this point → **Remove column from consideration**.
- `international_goals` in `players` has `min` of 0 but also about 61.4% of NULL values → **Needs exploring to see whether there is a difference between players with no international cap versus players with international caps but have scored zero goals versus players with no national cap data.**
- Both `market_value_in_eur` and `highest_market_value_in_eur` in `players` have the same NULL rates of ~17.19% → **Need checking whether the non-NULL entries are connected; as well as exploring the cohorts of players (academy players who still haven't had a markvet valuation, or (near-)retired players who don't need a player valuation).**
- The `*_position` columns in both `club_games` and `games` has the same NULL rate of ~28.53% → **Needs exploring further to see whether those games with NULL entries are cup games (which usually have no league position) or just low-coverage.**
- Are NULL values clustered together (by club, by competition, by country) or spread out.

##### Irregular values

- `minute` in `game_events` has min value of `-1` → **Needs exploring further to see the reason (normal minutes are usually non-negative, so -1 might be a placeholder)**.
- `own_goals` and  `opponent_goals` have near-identical general statistics → **Needs exploring further to see whether there between any difference in these two colunms or are they just copies of one another**.
- Many features have irregular minimal/maximum values:
    - Minimum of `average_age` in `clubs` is 0.
    - Minimum of `height_in_cm` in `players` is 17.
    - Minimum of `stadium_seats` in `clubs` is 0.
    - Minimum of `attendance` in `games` is 1.
    - Maximum of `minutes_played` in `appearances` is 148.

#### Data Exploration (1)

##### Question 1. Is `-1` in `minute` in `game_events` a placeholder?

Assumption: `-1` in `minute` in `game_events` is only a placeholder for a game with low-coverage (with minutes not logged)

In [11]:
nonpositive_min_events = con.execute("""
    SELECT *
    FROM game_events
    WHERE minute <= 0
""").df()

print(
    f"Number of game events with logged minute less or equal to 0: {len(nonpositive_min_events)}"
)
display(nonpositive_min_events)

print(
    f"Types of game events correlated with `minute` = -1: {list(nonpositive_min_events['type'].unique())}"
)

Number of game events with logged minute less or equal to 0: 17575


,game_event_id,date,game_id,minute,type,club_id,club_name,player_id,description,player_in_id,player_assist_id
0,587290140ebb0e436795bd4406fc6afe,2010-06-29,1027049,-1,Shootout,3581,Paraguay,2050,", Scored",<NA>,<NA>
1,6a7ec555f265921db64bf05304171894,2010-06-29,1027049,-1,Shootout,3581,Paraguay,16120,", Scored",<NA>,<NA>
2,0ed09be6d9c6643c6f05aa5a720221c6,2010-06-29,1027049,-1,Shootout,3435,Japan,27513,", Scored",<NA>,<NA>
3,b79cb0d0f6ae63bcd548ea4d64eb8640,2010-06-29,1027049,-1,Shootout,3581,Paraguay,37713,", Scored",<NA>,<NA>
4,80b2dbf5a7815aead1b85d7c414da362,2010-06-29,1027049,-1,Shootout,3435,Japan,38777,", Missed",<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...
17570,99d42bf1eed86e9140d4438b9ce0750d,2008-06-22,910151,-1,Shootout,3375,Spain,7840,", Scored",<NA>,<NA>
17571,1e9d8d9e1de7cb9772395e308a0b386e,2008-06-22,910151,-1,Shootout,3375,Spain,7980,", Scored",<NA>,<NA>
17572,e8be853ce02272d8b456b3c3d8340e0e,2008-06-22,910151,-1,Shootout,3375,Spain,8186,", Saved",<NA>,<NA>
17573,d33fd390a43bbedc563589c363c2408b,2008-06-22,910151,-1,Shootout,3375,Spain,8806,", Scored",<NA>,<NA>


Types of game events correlated with `minute` = -1: ['Shootout', 'Cards', 'Substitutions', 'Goals']


In [6]:
con.execute("""
    SELECT type,
           COUNT(*) AS n_events,
           COUNT(*) FILTER (WHERE minute = -1) AS n_neg_events,
           ROUND(100.0 * COUNT(*) FILTER (WHERE minute = -1) / COUNT(*), 4) AS pct_neg_events
    FROM game_events
    GROUP BY type
    ORDER BY n_neg_events DESC
""").df()

,type,n_events,n_neg_events,pct_neg_events
0,Shootout,13574,13574,100.0000
1,Cards,381753,3784,0.9912
2,Goals,247803,118,0.0476
3,Substitutions,631339,99,0.0157


- From the table, we can see that 100% of shootout goals has `minute=1`, which is explainable because shootouts are outside of official game time.
- For the other types of in-game events, they are few (< 1%), which might suggest a different reason for the `-1` placeholder → Might be cards issued after official time (due to on-field fighting, boycotts, or fan interruptions).

In [ ]:
con.execute("""
    WITH neg_events AS (
        SELECT game_id, COUNT(*) AS n_neg_cards
        FROM game_events ge
        WHERE ge.type = 'Cards' AND ge.minute = -1
        GROUP BY game_id
    ),
    per_game AS (
        SELECT game_id,
               COUNT(*) AS n_events,
               COUNT(*) FILTER (WHERE minute <> -1) AS n_normal
        FROM game_events
        GROUP BY game_id
    )
    SELECT COUNT(*) AS games_with_neg_cards,
           COUNT(*) FILTER (WHERE p.n_normal = 0) AS games_with_no_normal_events
    FROM neg_events n 
    JOIN per_game p 
    USING (game_id)
""").df()

,games_with_neg_cards,games_with_no_normal_events
0,1178,6


We can see only 6 games have zero coverage minute-by-minute on all event type compared to the 1178 games with a `minute` placeholder, thus support the theory more.

In [35]:
CATEGORICAL_TYPES = r"^(VARCHAR|BOOLEAN|ENUM.*)$"

summaries = []
for (table,) in db_tables:
    df = con.execute(f"""
        SELECT '{table}' AS table_name, s.*
        FROM (SUMMARIZE {table}) s
        WHERE regexp_matches(s.column_type, '{CATEGORICAL_TYPES}')
          AND s.column_name NOT LIKE '%!_id' ESCAPE '!'
    """).df()
    if not df.empty:
        summaries.append(df)

qualitative_features = pd.concat(summaries, ignore_index=True)
qualitative_features["unique_ratio"] = (
    qualitative_features["approx_unique"] / qualitative_features["count"]
).round(4)

display(
    qualitative_features[
        [
            "table_name",
            "column_name",
            "column_type",
            "approx_unique",
            "unique_ratio",
            "count",
            "null_percentage",
        ]
    ]
    .sort_values(by=["table_name", "approx_unique"])
    .reset_index(drop=True)
)

,table_name,column_name,column_type,approx_unique,unique_ratio,count,null_percentage
0,appearances,player_name,VARCHAR,28987,0.0153,1894350,0.00
1,club_games,hosting,VARCHAR,2,0.0000,177916,0.00
2,club_games,own_manager_name,VARCHAR,6550,0.0368,177916,0.95
3,club_games,opponent_manager_name,VARCHAR,6550,0.0368,177916,0.95
4,clubs,last_season,VARCHAR,12,0.0151,796,0.00
...,...,...,...,...,...,...,...
69,transfers,transfer_season,VARCHAR,34,0.0002,175165,0.00
70,transfers,to_club_name,VARCHAR,12593,0.0719,175165,0.00
71,transfers,from_club_name,VARCHAR,15260,0.0871,175165,0.00
72,transfers,player_name,VARCHAR,18558,0.1059,175165,0.00


In [38]:
TEMPORAL_TYPES = r"^(DATE|TIME.*|TIMESTAMP.*)$"

summaries = []
for (table,) in db_tables:
    df = con.execute(f"""
        SELECT '{table}' AS table_name, s.*
        FROM (SUMMARIZE {table}) s
        WHERE regexp_matches(s.column_type, '{TEMPORAL_TYPES}')
    """).df()
    if not df.empty:
        summaries.append(df)

temporal_features = pd.concat(summaries, ignore_index=True)
display(
    temporal_features[
        [
            "table_name",
            "column_name",
            "column_type",
            "min",
            "max",
            "approx_unique",
            "null_percentage",
        ]
    ].reset_index(drop=True)
)

,table_name,column_name,column_type,min,max,approx_unique,null_percentage
0,appearances,date,DATE,2012-07-03,2026-06-28,4345,0.00
1,games,date,DATE,2006-06-09,2026-07-06,4847,0.00
2,game_events,date,DATE,2006-06-09,2026-07-06,4847,0.00
3,game_lineups,date,DATE,2013-07-02,2026-07-06,4236,0.00
4,players,date_of_birth,TIMESTAMP,1968-07-31 00:00:00,2011-02-23 00:00:00,8460,0.10
5,players,contract_expiration_date,TIMESTAMP,2000-05-31 00:00:00,2035-06-30 00:00:00,228,37.03
6,player_valuations,date,DATE,2000-01-20,2026-06-12,5369,0.00
7,transfers,transfer_date,DATE,1993-07-01,2030-06-30,5316,0.00


In [6]:
con.close()